# Task 2.3 — SFT Model Evaluation

Compare the fine-tuned SFT model (LoRA adapter) vs the base DeepSeek-R1 on the
1,000-sample validation set.

| | |
|---|---|
| **Input** | `data/sft_val.jsonl` — 1,000 validation samples (chat-format JSONL) |
| **LoRA Adapter** | `models/sentiment_lora/` — LoRA adapter weights (1.05 GB) |
| **Base Model** | `deepseek-ai/DeepSeek-R1-Distill-Qwen-14B` |
| **Output** | `output/eval_report.json` — all metrics, breakdowns, error analysis |

> **Requires GPU**: A100 40GB on ACCRE.

## Step 1: Setup — Imports, Model Loading, Data Loading

In [ ]:
import json
import logging
import re
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ── Paths ────────────────────────────────────────────────────────────────
_cwd = Path.cwd()
VAL_DATA_PATH = _cwd / "data" / "sft_val.jsonl"
LORA_DIR = _cwd / "models" / "sentiment_lora"
OUTPUT_DIR = _cwd / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Constants ────────────────────────────────────────────────────────────
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
LABEL_ORDER = ["very_negative", "negative", "neutral", "positive", "very_positive"]
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.1

# ── Logging ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("task_2_3")

# ── Load Base Model (4-bit) ─────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading base model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

# ── Apply LoRA Adapter ──────────────────────────────────────────────────
print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(model, str(LORA_DIR))
model.eval()
print(f"Model loaded with LoRA adapter from {LORA_DIR}")

# ── Load Validation Data ────────────────────────────────────────────────
val_records = []
with open(VAL_DATA_PATH) as f:
    for line in f:
        val_records.append(json.loads(line.strip()))

# ── Extract True Labels & Metadata ──────────────────────────────────────
true_labels = []
meta_rows = []

for rec in val_records:
    # True label from assistant response
    assistant_msg = rec["messages"][2]["content"]
    parsed_label = json.loads(assistant_msg)["label"]
    true_labels.append(parsed_label)

    # Metadata from user message
    user_msg = rec["messages"][1]["content"]
    ticker_match = re.search(r"for (\w+) \(", user_msg)
    sub_sector_match = re.search(r"\((\w+(?:_\w+)*) sub-sector\)", user_msg)
    category_match = re.search(r"Category: (.+)", user_msg)
    factor_match = re.search(r"Factor: (.+)", user_msg)

    meta_rows.append({
        "ticker": ticker_match.group(1) if ticker_match else "UNKNOWN",
        "sub_sector": sub_sector_match.group(1) if sub_sector_match else "unknown",
        "category": category_match.group(1).strip() if category_match else "unknown",
        "factor_key": factor_match.group(1).strip() if factor_match else "unknown",
    })

meta_df = pd.DataFrame(meta_rows)
meta_df["true_label"] = true_labels

# ── Summary ──────────────────────────────────────────────────────────────
print(f"\nValidation samples: {len(val_records)}")
print(f"Label distribution:")
for label in LABEL_ORDER:
    count = true_labels.count(label)
    print(f"  {label:16s}: {count:4d} ({count/len(true_labels)*100:.1f}%)")
print(f"\nSub-sectors: {meta_df['sub_sector'].value_counts().to_dict()}")
print(f"Categories:  {meta_df['category'].nunique()} unique")
print(f"Tickers:     {meta_df['ticker'].nunique()} unique")

## Step 2: Inference Utilities

Key design decisions:
- **`<think>\n</think>\n` suffix**: Forces DeepSeek-R1 to skip chain-of-thought reasoning and produce JSON directly.
- **`add_special_tokens=False`**: Chat template already adds BOS token; prevents double-BOS.
- **Single-sample inference**: No batching — simpler and more reliable with variable-length inputs on a 4-bit model.
- **Parse failures**: Excluded from F1 but tracked as "JSON compliance rate."

In [ ]:
def parse_json_response(text: str) -> dict | None:
    """Parse JSON from model output, handling think blocks and markdown fences."""
    # Strip <think>...</think> block if present
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

    # Strip markdown code fences
    text = re.sub(r"^```(?:json)?\s*\n?", "", text, flags=re.MULTILINE)
    text = re.sub(r"\n?```\s*$", "", text, flags=re.MULTILINE)
    text = text.strip()

    # Try direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Fallback: find first { to last }
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(text[start : end + 1])
        except json.JSONDecodeError:
            pass

    return None


def extract_prediction(parsed: dict | None) -> tuple[str, str, float]:
    """Extract (label, rationale, confidence) from parsed JSON.
    
    Returns ("parse_error", "", 0.0) if parsing failed.
    """
    if parsed is None:
        return ("parse_error", "", 0.0)

    # Check both "label" and "sentiment" keys
    label = parsed.get("label") or parsed.get("sentiment", "")
    label = label.strip().lower()

    if label not in LABEL_ORDER:
        return ("parse_error", parsed.get("rationale", ""), 0.0)

    rationale = parsed.get("rationale", "")
    confidence = float(parsed.get("confidence", 0.0))

    return (label, rationale, confidence)


def run_inference(
    model,
    tokenizer,
    val_records: list[dict],
    desc: str = "Inference",
) -> list[dict]:
    """Run single-sample inference on all validation records.
    
    Returns list of dicts with: pred_label, rationale, confidence, raw_output.
    """
    import time

    results = []
    n = len(val_records)
    t0 = time.time()

    for i, rec in enumerate(val_records):
        messages = rec["messages"][:2]  # system + user only

        # Build input with empty think block to skip reasoning
        input_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        input_text += "<think>\n</think>\n"

        inputs = tokenizer(
            input_text, return_tensors="pt", add_special_tokens=False
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
            )

        # Decode only generated tokens
        generated = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1] :],
            skip_special_tokens=True,
        )

        parsed = parse_json_response(generated)
        label, rationale, confidence = extract_prediction(parsed)

        results.append({
            "pred_label": label,
            "rationale": rationale,
            "confidence": confidence,
            "raw_output": generated,
        })

        # Progress update every sample
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed
        eta = (n - i - 1) / rate if rate > 0 else 0
        print(
            f"\r {desc}: [{i+1}/{n}] "
            f"{elapsed:.0f}s elapsed, {eta:.0f}s remaining, "
            f"{rate:.2f} it/s",
            end="", flush=True,
        )

        # Free GPU cache periodically
        if (i + 1) % 100 == 0:
            torch.cuda.empty_cache()

    elapsed = time.time() - t0
    print(f"\n {desc} complete: {n} samples in {elapsed:.0f}s ({elapsed/60:.1f} min)")
    return results


print("Inference utilities defined: parse_json_response, extract_prediction, run_inference")

## Step 3: SFT Model Inference (LoRA Enabled)

Run the fine-tuned model on all 1,000 validation samples. Expected runtime ~8–15 min on A100.

In [ ]:
model.enable_adapter_layers()

sft_results = run_inference(model, tokenizer, val_records, desc="SFT inference")

# Parse success rate
sft_parse_ok = sum(1 for r in sft_results if r["pred_label"] != "parse_error")
sft_parse_fail = len(sft_results) - sft_parse_ok

print(f"\nSFT inference complete:")
print(f"  Parsed successfully: {sft_parse_ok}/{len(sft_results)} ({sft_parse_ok/len(sft_results)*100:.1f}%)")
print(f"  Parse failures:     {sft_parse_fail}")

if sft_parse_fail > 0:
    fail_idx = [i for i, r in enumerate(sft_results) if r["pred_label"] == "parse_error"]
    print(f"\n  Sample failure outputs (first 3):")
    for idx in fail_idx[:3]:
        print(f"    [{idx}] {sft_results[idx]['raw_output'][:150]}...")

## Step 4: Base Model Inference (LoRA Disabled)

Run the same validation samples through the base model (LoRA adapters disabled) to establish a baseline for comparison.

In [ ]:
model.disable_adapter_layers()

base_results = run_inference(model, tokenizer, val_records, desc="Base inference")

# Parse success rate
base_parse_ok = sum(1 for r in base_results if r["pred_label"] != "parse_error")
base_parse_fail = len(base_results) - base_parse_ok

print(f"\nBase inference complete:")
print(f"  Parsed successfully: {base_parse_ok}/{len(base_results)} ({base_parse_ok/len(base_results)*100:.1f}%)")
print(f"  Parse failures:     {base_parse_fail}")

if base_parse_fail > 0:
    fail_idx = [i for i, r in enumerate(base_results) if r["pred_label"] == "parse_error"]
    print(f"\n  Sample failure outputs (first 3):")
    for idx in fail_idx[:3]:
        print(f"    [{idx}] {base_results[idx]['raw_output'][:150]}...")

# Re-enable LoRA for any later use
model.enable_adapter_layers()

## Step 5: Classification Metrics

Compute per-class precision, recall, F1 for both models. Parse failures are excluded from F1 but tracked separately as JSON compliance rate.

In [ ]:
# ── Filter to parseable predictions ──────────────────────────────────────
sft_mask = [r["pred_label"] != "parse_error" for r in sft_results]
base_mask = [r["pred_label"] != "parse_error" for r in base_results]

sft_true = [t for t, m in zip(true_labels, sft_mask) if m]
sft_pred = [r["pred_label"] for r, m in zip(sft_results, sft_mask) if m]

base_true = [t for t, m in zip(true_labels, base_mask) if m]
base_pred = [r["pred_label"] for r, m in zip(base_results, base_mask) if m]

# ── SFT Classification Report ───────────────────────────────────────────
print("=" * 70)
print("SFT MODEL — Classification Report")
print("=" * 70)
print(classification_report(sft_true, sft_pred, labels=LABEL_ORDER, digits=3, zero_division=0))

sft_report = classification_report(
    sft_true, sft_pred, labels=LABEL_ORDER, output_dict=True, zero_division=0
)

# ── Base Classification Report ───────────────────────────────────────────
print("=" * 70)
print("BASE MODEL — Classification Report")
print("=" * 70)
print(classification_report(base_true, base_pred, labels=LABEL_ORDER, digits=3, zero_division=0))

base_report = classification_report(
    base_true, base_pred, labels=LABEL_ORDER, output_dict=True, zero_division=0
)

# ── Summary Metrics ──────────────────────────────────────────────────────
sft_metrics = {
    "accuracy": accuracy_score(sft_true, sft_pred),
    "macro_f1": f1_score(sft_true, sft_pred, labels=LABEL_ORDER, average="macro", zero_division=0),
    "weighted_f1": f1_score(sft_true, sft_pred, labels=LABEL_ORDER, average="weighted", zero_division=0),
    "json_compliance": sum(sft_mask) / len(sft_mask),
    "n_parseable": sum(sft_mask),
}

base_metrics = {
    "accuracy": accuracy_score(base_true, base_pred),
    "macro_f1": f1_score(base_true, base_pred, labels=LABEL_ORDER, average="macro", zero_division=0),
    "weighted_f1": f1_score(base_true, base_pred, labels=LABEL_ORDER, average="weighted", zero_division=0),
    "json_compliance": sum(base_mask) / len(base_mask),
    "n_parseable": sum(base_mask),
}

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"{'Metric':<20s} {'SFT':>10s} {'Base':>10s} {'Delta':>10s}")
print("-" * 50)
for key in ["accuracy", "macro_f1", "weighted_f1", "json_compliance"]:
    s = sft_metrics[key]
    b = base_metrics[key]
    d = s - b
    sign = "+" if d >= 0 else ""
    print(f"{key:<20s} {s:>10.3f} {b:>10.3f} {sign}{d:>9.3f}")

## Step 6: SFT vs Base — Per-Class Comparison Table

In [ ]:
# ── Per-class F1 comparison table ────────────────────────────────────────
rows = []
for label in LABEL_ORDER:
    sft_f1 = sft_report.get(label, {}).get("f1-score", 0.0)
    base_f1 = base_report.get(label, {}).get("f1-score", 0.0)
    rows.append({
        "Label": label,
        "SFT Prec": sft_report.get(label, {}).get("precision", 0.0),
        "SFT Rec": sft_report.get(label, {}).get("recall", 0.0),
        "SFT F1": sft_f1,
        "Base F1": base_f1,
        "Delta F1": sft_f1 - base_f1,
    })

# Add macro avg row
rows.append({
    "Label": "macro avg",
    "SFT Prec": sft_report["macro avg"]["precision"],
    "SFT Rec": sft_report["macro avg"]["recall"],
    "SFT F1": sft_report["macro avg"]["f1-score"],
    "Base F1": base_report["macro avg"]["f1-score"],
    "Delta F1": sft_report["macro avg"]["f1-score"] - base_report["macro avg"]["f1-score"],
})

# Add accuracy and JSON compliance
rows.append({
    "Label": "accuracy",
    "SFT Prec": None,
    "SFT Rec": None,
    "SFT F1": sft_metrics["accuracy"],
    "Base F1": base_metrics["accuracy"],
    "Delta F1": sft_metrics["accuracy"] - base_metrics["accuracy"],
})
rows.append({
    "Label": "JSON compliance",
    "SFT Prec": None,
    "SFT Rec": None,
    "SFT F1": sft_metrics["json_compliance"],
    "Base F1": base_metrics["json_compliance"],
    "Delta F1": sft_metrics["json_compliance"] - base_metrics["json_compliance"],
})

comparison_df = pd.DataFrame(rows).set_index("Label")

# Style: highlight positive deltas green, negative red
def highlight_delta(val):
    if pd.isna(val):
        return ""
    if val > 0.005:
        return "color: green; font-weight: bold"
    elif val < -0.005:
        return "color: red"
    return ""

styled = (
    comparison_df.style
    .format("{:.3f}", na_rep="—")
    .applymap(highlight_delta, subset=["Delta F1"])
)
display(styled)

## Step 7: Confusion Matrices

Side-by-side confusion matrices for SFT (left) and Base (right) models.

In [ ]:
SHORT_LABELS = ["v_neg", "neg", "neut", "pos", "v_pos"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# SFT confusion matrix
cm_sft = confusion_matrix(sft_true, sft_pred, labels=LABEL_ORDER)
sns.heatmap(
    cm_sft, annot=True, fmt="d", cmap="Blues",
    xticklabels=SHORT_LABELS, yticklabels=SHORT_LABELS, ax=ax1,
)
ax1.set_title("SFT Model — Confusion Matrix", fontsize=13)
ax1.set_xlabel("Predicted")
ax1.set_ylabel("True")

# Base confusion matrix
cm_base = confusion_matrix(base_true, base_pred, labels=LABEL_ORDER)
sns.heatmap(
    cm_base, annot=True, fmt="d", cmap="Blues",
    xticklabels=SHORT_LABELS, yticklabels=SHORT_LABELS, ax=ax2,
)
ax2.set_title("Base Model — Confusion Matrix", fontsize=13)
ax2.set_xlabel("Predicted")
ax2.set_ylabel("True")

plt.tight_layout()
plt.show()

# ── Identify most confused pairs ────────────────────────────────────────
print("\nMost confused label pairs (SFT model):")
# Zero out diagonal, find top off-diagonal entries
cm_off = cm_sft.copy()
np.fill_diagonal(cm_off, 0)
for _ in range(3):
    idx = np.unravel_index(cm_off.argmax(), cm_off.shape)
    true_l = LABEL_ORDER[idx[0]]
    pred_l = LABEL_ORDER[idx[1]]
    count = cm_off[idx]
    if count > 0:
        print(f"  {true_l} → {pred_l}: {count} samples")
    cm_off[idx] = 0

## Step 8: Sub-Sector Breakdown

Macro F1 per sub-sector for both models. Does SFT help more for some sub-sectors?

In [ ]:
# ── Build eval DataFrame with predictions ────────────────────────────────
eval_df = meta_df.copy()
eval_df["sft_pred"] = [r["pred_label"] for r in sft_results]
eval_df["base_pred"] = [r["pred_label"] for r in base_results]
eval_df["sft_conf"] = [r["confidence"] for r in sft_results]
eval_df["base_conf"] = [r["confidence"] for r in base_results]

# ── Sub-sector F1 breakdown ──────────────────────────────────────────────
subsector_rows = []
for ss, group in eval_df.groupby("sub_sector"):
    n = len(group)

    # SFT: filter parseable
    sft_ok = group[group["sft_pred"] != "parse_error"]
    if len(sft_ok) > 0:
        sft_f1 = f1_score(
            sft_ok["true_label"], sft_ok["sft_pred"],
            labels=LABEL_ORDER, average="macro", zero_division=0,
        )
    else:
        sft_f1 = 0.0

    # Base: filter parseable
    base_ok = group[group["base_pred"] != "parse_error"]
    if len(base_ok) > 0:
        base_f1 = f1_score(
            base_ok["true_label"], base_ok["base_pred"],
            labels=LABEL_ORDER, average="macro", zero_division=0,
        )
    else:
        base_f1 = 0.0

    subsector_rows.append({
        "sub_sector": ss,
        "n_samples": n,
        "sft_f1": sft_f1,
        "base_f1": base_f1,
        "delta_f1": sft_f1 - base_f1,
    })

subsector_df = pd.DataFrame(subsector_rows).sort_values("n_samples", ascending=False)
print(subsector_df.to_string(index=False, float_format="%.3f"))

# ── Grouped bar chart ────────────────────────────────────────────────────
x = np.arange(len(subsector_df))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width / 2, subsector_df["sft_f1"], width, label="SFT", color="steelblue")
bars2 = ax.bar(x + width / 2, subsector_df["base_f1"], width, label="Base", color="lightcoral")

ax.set_ylabel("Macro F1")
ax.set_title("Macro F1 by Sub-Sector")
ax.set_xticks(x)
ax.set_xticklabels(subsector_df["sub_sector"], rotation=15, ha="right")
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.3)

# Annotate sample counts
for i, row in enumerate(subsector_df.itertuples()):
    ax.text(i, max(row.sft_f1, row.base_f1) + 0.02, f"n={row.n_samples}",
            ha="center", fontsize=9, color="gray")

plt.tight_layout()
plt.show()

## Step 9: Category Breakdown

F1 per question category for both models. Categories with fewer than 20 samples are flagged as unreliable.

In [ ]:
# ── Category F1 breakdown ────────────────────────────────────────────────
cat_rows = []
for cat, group in eval_df.groupby("category"):
    n = len(group)

    sft_ok = group[group["sft_pred"] != "parse_error"]
    sft_f1 = (
        f1_score(sft_ok["true_label"], sft_ok["sft_pred"],
                 labels=LABEL_ORDER, average="macro", zero_division=0)
        if len(sft_ok) > 0 else 0.0
    )

    base_ok = group[group["base_pred"] != "parse_error"]
    base_f1 = (
        f1_score(base_ok["true_label"], base_ok["base_pred"],
                 labels=LABEL_ORDER, average="macro", zero_division=0)
        if len(base_ok) > 0 else 0.0
    )

    cat_rows.append({
        "category": cat,
        "n_samples": n,
        "sft_f1": sft_f1,
        "base_f1": base_f1,
        "delta_f1": sft_f1 - base_f1,
    })

cat_df = pd.DataFrame(cat_rows).sort_values("delta_f1", ascending=False)

# Flag small-sample categories
cat_df["reliable"] = cat_df["n_samples"] >= 20

print(cat_df.to_string(index=False, float_format="%.3f"))
print(f"\n* Categories with n < 20 may have unreliable F1 estimates.")

# ── Horizontal grouped bar chart ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, max(6, len(cat_df) * 0.4)))

y = np.arange(len(cat_df))
height = 0.35

ax.barh(y - height / 2, cat_df["sft_f1"], height, label="SFT", color="steelblue")
ax.barh(y + height / 2, cat_df["base_f1"], height, label="Base", color="lightcoral")

ax.set_xlabel("Macro F1")
ax.set_title("Macro F1 by Category (sorted by delta)")
ax.set_yticks(y)
ax.set_yticklabels(cat_df["category"], fontsize=9)
ax.legend(loc="lower right")
ax.set_xlim(0, 1.05)
ax.grid(axis="x", alpha=0.3)

# Mark unreliable categories
for i, row in enumerate(cat_df.itertuples()):
    if not row.reliable:
        ax.text(1.02, i, f"n={row.n_samples} *", fontsize=8, color="orange", va="center")
    else:
        ax.text(1.02, i, f"n={row.n_samples}", fontsize=8, color="gray", va="center")

plt.tight_layout()
plt.show()

## Step 10: Error Analysis

Sample 20 SFT misclassifications and analyze error patterns: adjacent (distance 1) vs distant (distance 2+), most common error pairs, and confidence on correct vs incorrect predictions.

In [ ]:
# ── Identify SFT errors ──────────────────────────────────────────────────
label_to_idx = {l: i for i, l in enumerate(LABEL_ORDER)}

sft_errors = eval_df[
    (eval_df["sft_pred"] != "parse_error")
    & (eval_df["sft_pred"] != eval_df["true_label"])
].copy()

sft_errors["distance"] = sft_errors.apply(
    lambda r: abs(label_to_idx[r["true_label"]] - label_to_idx[r["sft_pred"]]), axis=1
)
sft_errors["severity"] = sft_errors["distance"].apply(
    lambda d: "adjacent" if d == 1 else "distant"
)

print(f"Total SFT errors: {len(sft_errors)} / {sum(sft_mask)} parseable ({len(sft_errors)/sum(sft_mask)*100:.1f}%)")
print(f"  Adjacent (distance 1): {(sft_errors['severity'] == 'adjacent').sum()}")
print(f"  Distant  (distance 2+): {(sft_errors['severity'] == 'distant').sum()}")

# ── Sample 20 misclassifications ─────────────────────────────────────────
sample_n = min(20, len(sft_errors))
sample_errors = sft_errors.sample(n=sample_n, random_state=42)

print(f"\n{'='*80}")
print(f"Sample {sample_n} SFT Misclassifications")
print(f"{'='*80}")

for i, (idx, row) in enumerate(sample_errors.iterrows()):
    user_msg = val_records[idx]["messages"][1]["content"]
    # Extract summary from user message
    summary_match = re.search(r"Summary: (.+?)(?:\nEvidence:)", user_msg, re.DOTALL)
    summary_text = summary_match.group(1).strip()[:200] if summary_match else "N/A"

    print(f"\n[{i+1}] {row['ticker']} | {row['sub_sector']} | {row['category']}")
    print(f"    Factor: {row['factor_key']}")
    print(f"    Summary: {summary_text}")
    print(f"    True: {row['true_label']}  →  Pred: {row['sft_pred']}  (distance={row['distance']}, conf={row['sft_conf']:.2f})")

# ── Aggregate error patterns ─────────────────────────────────────────────
print(f"\n{'='*80}")
print("Error Pattern Aggregates")
print(f"{'='*80}")

# Most common error pairs
print("\nTop 5 error pairs (true → pred):")
error_pairs = sft_errors.groupby(["true_label", "sft_pred"]).size().sort_values(ascending=False)
for (true_l, pred_l), count in error_pairs.head(5).items():
    print(f"  {true_l:16s} → {pred_l:16s}: {count}")

# Error rate by sub-sector
print("\nError rate by sub-sector:")
for ss, group in eval_df[eval_df["sft_pred"] != "parse_error"].groupby("sub_sector"):
    n_errors = (group["sft_pred"] != group["true_label"]).sum()
    n_total = len(group)
    print(f"  {ss:25s}: {n_errors}/{n_total} ({n_errors/n_total*100:.1f}%)")

# Error rate by category
print("\nError rate by category (top 5 worst):")
cat_errors = []
for cat, group in eval_df[eval_df["sft_pred"] != "parse_error"].groupby("category"):
    n_errors = (group["sft_pred"] != group["true_label"]).sum()
    n_total = len(group)
    cat_errors.append({"category": cat, "error_rate": n_errors / n_total, "n_errors": n_errors, "n_total": n_total})
cat_error_df = pd.DataFrame(cat_errors).sort_values("error_rate", ascending=False)
for _, row in cat_error_df.head(5).iterrows():
    print(f"  {row['category']:30s}: {row['n_errors']:.0f}/{row['n_total']:.0f} ({row['error_rate']*100:.1f}%)")

# Mean confidence: correct vs incorrect
correct_mask = eval_df["sft_pred"] == eval_df["true_label"]
parseable_mask = eval_df["sft_pred"] != "parse_error"

mean_conf_correct = eval_df.loc[correct_mask & parseable_mask, "sft_conf"].mean()
mean_conf_incorrect = eval_df.loc[~correct_mask & parseable_mask, "sft_conf"].mean()
print(f"\nMean confidence — correct:   {mean_conf_correct:.3f}")
print(f"Mean confidence — incorrect: {mean_conf_incorrect:.3f}")
print(f"Delta:                       {mean_conf_correct - mean_conf_incorrect:+.3f}")

## Step 11: Save Evaluation Report

Save all metrics, per-class/sub-sector/category breakdowns, and error analysis to `output/eval_report.json`.

In [ ]:
# ── Build report structure ────────────────────────────────────────────────
report = {
    "metadata": {
        "base_model": MODEL_NAME,
        "lora_dir": str(LORA_DIR),
        "val_samples": len(val_records),
        "timestamp": datetime.now().isoformat(),
    },
    "sft": {
        "accuracy": sft_metrics["accuracy"],
        "macro_f1": sft_metrics["macro_f1"],
        "weighted_f1": sft_metrics["weighted_f1"],
        "json_compliance": sft_metrics["json_compliance"],
        "n_parseable": sft_metrics["n_parseable"],
        "per_class": {
            label: {
                "precision": sft_report[label]["precision"],
                "recall": sft_report[label]["recall"],
                "f1": sft_report[label]["f1-score"],
                "support": sft_report[label]["support"],
            }
            for label in LABEL_ORDER
        },
    },
    "base": {
        "accuracy": base_metrics["accuracy"],
        "macro_f1": base_metrics["macro_f1"],
        "weighted_f1": base_metrics["weighted_f1"],
        "json_compliance": base_metrics["json_compliance"],
        "n_parseable": base_metrics["n_parseable"],
        "per_class": {
            label: {
                "precision": base_report[label]["precision"],
                "recall": base_report[label]["recall"],
                "f1": base_report[label]["f1-score"],
                "support": base_report[label]["support"],
            }
            for label in LABEL_ORDER
        },
    },
    "comparison": {
        "delta_accuracy": sft_metrics["accuracy"] - base_metrics["accuracy"],
        "delta_macro_f1": sft_metrics["macro_f1"] - base_metrics["macro_f1"],
        "delta_weighted_f1": sft_metrics["weighted_f1"] - base_metrics["weighted_f1"],
        "delta_json_compliance": sft_metrics["json_compliance"] - base_metrics["json_compliance"],
        "per_class_delta_f1": {
            label: sft_report[label]["f1-score"] - base_report[label]["f1-score"]
            for label in LABEL_ORDER
        },
    },
    "sub_sector_breakdown": subsector_df.to_dict(orient="records"),
    "category_breakdown": cat_df.drop(columns=["reliable"]).to_dict(orient="records"),
    "error_analysis": {
        "total_errors": len(sft_errors),
        "adjacent_errors": int((sft_errors["severity"] == "adjacent").sum()),
        "distant_errors": int((sft_errors["severity"] == "distant").sum()),
        "mean_confidence_correct": float(mean_conf_correct),
        "mean_confidence_incorrect": float(mean_conf_incorrect),
        "top_error_pairs": [
            {"true": true_l, "pred": pred_l, "count": int(count)}
            for (true_l, pred_l), count in error_pairs.head(5).items()
        ],
    },
}

# ── Save ─────────────────────────────────────────────────────────────────
report_path = OUTPUT_DIR / "eval_report.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"Evaluation report saved to: {report_path}")
print(f"File size: {report_path.stat().st_size / 1024:.1f} KB")

# ── Headline metrics ─────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("HEADLINE METRICS")
print(f"{'='*60}")
print(f"SFT Macro F1:        {sft_metrics['macro_f1']:.3f}")
print(f"Base Macro F1:       {base_metrics['macro_f1']:.3f}")
print(f"Improvement:         {sft_metrics['macro_f1'] - base_metrics['macro_f1']:+.3f}")
print(f"SFT Accuracy:        {sft_metrics['accuracy']:.3f}")
print(f"SFT JSON Compliance: {sft_metrics['json_compliance']:.1%}")
print(f"Base JSON Compliance:{base_metrics['json_compliance']:.1%}")